In [ ]:
import pandas as pd
import plotly.express as px
from scipy.spatial import ConvexHull
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.kaleido.scope.default_format = "svg"

def plot__histogram(df, title=None,figure="figure"):
    if title is None:
        title = f"CPU Cycles vs Latency by Architecture ({df['arch'].nunique()} Architectures)"
    #print(title)
    # Create a Plotly scatter plot
    #fig = px.line(df, x="cycles", y="latency", text="arch", title="Cycles vs Latency", labels={"cycles": "CPU Cycles", "latency": "Latency"})
    #fig = px.scatter(df, y="cycles", x="latency", color="arch", title="Cycles vs Latency by Architecture", labels={"cycles": "CPU Cycles", "latency": "Latency", "arch": "Architecture"})
    #fig = px.histogram(df, x="latency", y="cycles", color="arch", barmode="group", title="Histogram of Cycles vs Latency by Architecture", labels={"cycles": "CPU Cycles", "latency": "Latency", "arch": "Architecture"})
    fig = px.bar(df, x="Instruction memory latency", y="System Cycles", color="arch", barmode='group', labels={"cycles": "System Cycles", "latency": "Instruction Memory Latency", "arch": "Architecture"}, title=title)
    #fig = px.bar(df, x="latency", y="cycles", color="arch", barmode='group', labels={"cycles": "CPU Cycles", "latency": "Latency", "arch": "Architecture"})
    # Force x-axis labels to be exact values from 'latency' column
    fig.update_xaxes(type='category', categoryorder='array', categoryarray=sorted(df['latency'].unique()))
    fig.show()
    fig.write_image(f"{figure}.pdf")
    fig.write_image(f"{figure}.svg")
# Print the DataFrame
#print(df)

def diff(df1,df2, title,figure):


    # Concatenate the DataFrames
    df = pd.concat([df1, df2], ignore_index=True)

    lca_cycles = df[df['arch'] == 'LCA'].set_index('latency')['cycles'].to_dict()


    # Create a dictionary mapping latency to LCA cycles
    df['cycles'] = df.apply(
        lambda row: row['cycles'] / lca_cycles.get(row['latency'], row['cycles']),
        axis=1
    )
    plot__histogram(df,title=title,figure=figure)

# onnx_conv2d

In [ ]:
import pandas as pd

file1 = "conv2d_test.csv"
file2 = "conv2d_test_lca.csv"

df1 = pd.read_csv(file1)
df1['arch'] = df1['arch'].apply(lambda x: "TCA-XREG")
df2 = pd.read_csv(file2)

# Concatenate the DataFrames
df = pd.concat([df2, df1], ignore_index=True)

lca_cycles = df[df['arch'] == 'LCA'].set_index('latency')['cycles'].to_dict()

# Save to a new CSV file (optional)
#merged_df.to_csv("merged_file.csv", index=False)

# Create a dictionary mapping latency to LCA cycles
df['cycles'] = df.apply(
    lambda row: row['cycles'] / lca_cycles.get(row['latency'], row['cycles']),
    axis=1
)


#print(df)
plot__histogram(df,title="onnx_conv2d",figure="onnx_conv2d")


In [ ]:
file1 = "redmule_complex_128b.csv"
file2 = "redmule_complex_ex_reg.csv"

df1 = pd.read_csv(file1)
df1['arch'] = df1['arch'].apply(lambda x: "TCA, enc=128b") 
df2 = pd.read_csv(file2)
df2['arch'] = df2['arch'].apply(lambda x: "TCA-XREG, enc=32b")

df = pd.concat([df2, df1], ignore_index=True)

lca_cycles = df[df['arch'] == 'TCA-XREG, enc=32b'].set_index('latency')['cycles'].to_dict()

# Save to a new CSV file (optional)
#merged_df.to_csv("merged_file.csv", index=False)

# Create a dictionary mapping latency to LCA cycles
df['cycles'] = df.apply(
    lambda row: row['cycles'] / lca_cycles.get(row['latency'], row['cycles']),
    axis=1
)


#print(df)
plot__histogram(df,title="redmule.gemm with/without Exchange Register File (XREG)",figure="redmule.gemm.xreg")

In [ ]:
file1 = "redmule_complex_128b.csv"
file2 = "redmule_complex_ex_reg.csv"
file3 = "redmule_complex_32b.csv"
file4 = "redmule.csv"

df1 = pd.read_csv(file1)
df1['arch'] = df1['arch'].apply(lambda x: "TCA, enc=128b") 

df2 = pd.read_csv(file2)
df2['arch'] = df2['arch'].apply(lambda x: "TCA-XREG, enc=32b")

df3 = pd.read_csv(file3)
df3['arch'] = df3['arch'].apply(lambda x: "TCA, enc=32b") 

df_lca = pd.read_csv(file4)
df_lca = df_lca[df_lca['id'] == 1]

df_tca = pd.concat([df3, df2, df1], ignore_index=True)

diff(df_lca,df_tca,title="GEMM with RedMulE acceleration",figure="redmule.gemm.all")
#print(df_lca)
